In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to cache file
cache_file = "cache_all_slides.pkl"

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
from helper_functions import subset_df, subset_df_list

df_HE = subset_df(df_all, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
df_HE["T_category"].value_counts().head()

In [ ]:
df_HE = subset_df_list(df_HE, "T_text", "Placenta")

In [ ]:
# Feature Results
feature_result = r"D:\NOTEBOOKS\Christine\all_slides\feature_summary_old.csv"
df_feature_result = pd.read_csv(feature_result)
df_feature_result = df_feature_result[df_feature_result['status'] == "feature extraction complete"].copy()
df_features = df_feature_result[df_feature_result['model'] == "h-optimus-0"].copy()
with_features = set(df_features["wsi_path"])
print(len(with_features))

df_HE = df_HE[df_HE["filename"].isin(with_features)]
print(len(df_HE))

In [ ]:
# Subset with 2 samples per rekvnr
sub_rekvnr = df_HE["rekvnr"].value_counts()
sub_rekvnr = sub_rekvnr[sub_rekvnr >= 2].sample(n=5, random_state=42).index

df_HE = (
    df_HE[df_HE["rekvnr"].isin(sub_rekvnr)]
    .groupby("rekvnr", group_keys=False)
    .sample(n=2, random_state=42)
)
print(df_HE)

In [ ]:
all_filenames = df_HE["filename"].tolist()
print("Number of wsi filenames: ", len(all_filenames))

In [ ]:
# Save to csv
output_file = "D:\DATA\spatial_domain_exp1_placenta.csv"

df_HE.to_csv(output_file, index=False)
print(f"Saved DataFrame to {output_file}")